In [19]:
A   = 'CTAGGATCCAGGCATACGA'
B = 'GGATCCATTCATTA'
K = 4
X = 2
MATCH, MISMATCH = 1, -1

def score(a, b):
    return MATCH if a == b else MISMATCH

# --- Индекс БД ---
index = {}
for i in range(len(A) - K + 1):
    kmer = A[i:i+K]
    index.setdefault(kmer, []).append(i)

print(' Индекс БД')
for kmer, positions in sorted(index.items()):
    print(f'  {kmer}: {positions}')

# --- Поиск seeds ---
seeds = []
for i in range(len(B) - K + 1):
    kmer = B[i:i+K]
    if kmer in index:
        for d in index[kmer]:
            seeds.append((i, d, kmer))

print(f'Seeds')
for q, d, kmer in seeds:
    print(f"  '{kmer}' → Q[{q}] D[{d}]")

 Индекс БД
  ACGA: [15]
  AGGA: [2]
  AGGC: [9]
  ATAC: [13]
  ATCC: [5]
  CAGG: [8]
  CATA: [12]
  CCAG: [7]
  CTAG: [0]
  GATC: [4]
  GCAT: [11]
  GGAT: [3]
  GGCA: [10]
  TACG: [14]
  TAGG: [1]
  TCCA: [6]
Seeds
  'GGAT' → Q[0] D[3]
  'GATC' → Q[1] D[4]
  'ATCC' → Q[2] D[5]
  'TCCA' → Q[3] D[6]


In [20]:
def extend(q_start, d_start, direction, seed_score):
    Scur, Smax = seed_score, seed_score
    best_q, best_d = q_start, d_start
    log = []
    q, d = q_start, d_start

    while True:
        q += direction
        d += direction
        if not (0 <= q < len(B) and 0 <= d < len(A)):
            break
        Scur += score(B[q], A[d])
        log.append((q, d, B[q], A[d], Scur))
        if Scur > Smax:
            Smax, best_q, best_d = Scur, q, d
        if Smax - Scur >= X:
            break

    return best_q, best_d, Smax, log


best_result = None

for q_pos, d_pos, kmer in seeds:
    seed_score = K * MATCH
    print(f"Seed: '{kmer}' | Q[{q_pos}] ↔ D[{d_pos}] | seed_score={seed_score}")

    bq_l, bd_l, smax_l, log_l = extend(q_pos, d_pos, -1, seed_score)
    print(f'  Расширение ВЛЕВО:')
    for (q,d,qc,dc,sc) in log_l:
        print(f'    Q[{q}]={qc} D[{d}]={dc}  Scur={sc}')
    if not log_l:
        print('    (граница — шагов нет)')
    print(f'  Smax_left={smax_l}  лучшая позиция: Q[{bq_l}] D[{bd_l}]')

    q_right = q_pos + K - 1
    d_right = d_pos + K - 1
    bq_r, bd_r, smax_r, log_r = extend(q_right, d_right, +1, seed_score)
    print(f'  Расширение ВПРАВО:')
    for (q,d,qc,dc,sc) in log_r:
        print(f'    Q[{q}]={qc} D[{d}]={dc}  Scur={sc}')
    if not log_r:
        print('    (граница — шагов нет)')
    print(f'  Smax_right={smax_r}  лучшая позиция: Q[{bq_r}] D[{bd_r}]')

    total = smax_l + smax_r - seed_score
    q_aln = B[bq_l:bq_r+1]
    d_aln = A[bd_l:bd_r+1]
    print(f'  Итоговый Score = {smax_l} + {smax_r} - {seed_score} = {total}')
    print(f'  B   [{bq_l}..{bq_r}]: {q_aln}')
    print(f'  A     [{bd_l}..{bd_r}]: {d_aln}')

    if best_result is None or total > best_result['total']:
        best_result = dict(
            kmer=kmer, total=total,
            q_aln=q_aln, d_aln=d_aln,
            bq_l=bq_l, bq_r=bq_r, bd_l=bd_l, bd_r=bd_r
        )

Seed: 'GGAT' | Q[0] ↔ D[3] | seed_score=4
  Расширение ВЛЕВО:
    (граница — шагов нет)
  Smax_left=4  лучшая позиция: Q[0] D[3]
  Расширение ВПРАВО:
    Q[4]=C D[7]=C  Scur=5
    Q[5]=C D[8]=C  Scur=6
    Q[6]=A D[9]=A  Scur=7
    Q[7]=T D[10]=G  Scur=6
    Q[8]=T D[11]=G  Scur=5
  Smax_right=7  лучшая позиция: Q[6] D[9]
  Итоговый Score = 4 + 7 - 4 = 7
  B   [0..6]: GGATCCA
  A     [3..9]: GGATCCA
Seed: 'GATC' | Q[1] ↔ D[4] | seed_score=4
  Расширение ВЛЕВО:
    Q[0]=G D[3]=G  Scur=5
  Smax_left=5  лучшая позиция: Q[0] D[3]
  Расширение ВПРАВО:
    Q[5]=C D[8]=C  Scur=5
    Q[6]=A D[9]=A  Scur=6
    Q[7]=T D[10]=G  Scur=5
    Q[8]=T D[11]=G  Scur=4
  Smax_right=6  лучшая позиция: Q[6] D[9]
  Итоговый Score = 5 + 6 - 4 = 7
  B   [0..6]: GGATCCA
  A     [3..9]: GGATCCA
Seed: 'ATCC' | Q[2] ↔ D[5] | seed_score=4
  Расширение ВЛЕВО:
    Q[1]=G D[4]=G  Scur=5
    Q[0]=G D[3]=G  Scur=6
  Smax_left=6  лучшая позиция: Q[0] D[3]
  Расширение ВПРАВО:
    Q[6]=A D[9]=A  Scur=5
    Q[7]=T D[10]=G

In [21]:
b = best_result
mid = ''.join('|' if a==c else ' ' for a,c in zip(b['d_aln'], b['q_aln']))

print('лучшее выравнивание')
print(f"k-мер: '{b['kmer']}'")
print(f"Smax = {b['total']}")
print(f"A    [{b['bd_l']}..{b['bd_r']}]: {b['d_aln']}")
print(f"               {mid}")
print(f"B [{b['bq_l']}..{b['bq_r']}]: {b['q_aln']}")

лучшее выравнивание
k-мер: 'GGAT'
Smax = 7
A    [3..9]: GGATCCA
               |||||||
B [0..6]: GGATCCA
